In [4]:
import os
import glob
import random
from pathlib import Path
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
from PIL import Image
import pandas as pd
import urllib.request
import json


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
model.eval()

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cuda


AttributeError: module 'torch' has no attribute '_utils'

In [ ]:

def load_imagenet_mapping():
    """
    Downloads the official ImageNet class index from a stable AWS mirror
    and creates a mapping from WordNet IDs (n0...) to ResNet integers.
    """
    print("Downloading ImageNet class mapping...")
    url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
    
    response = urllib.request.urlopen(url)
    class_idx = json.loads(response.read())
    wordnet_to_idx = {v[0]: int(k) for k, v in class_idx.items()}
    
    print(f"Successfully loaded {len(wordnet_to_idx)} class mappings!")
    return wordnet_to_idx

ORACLE_DICTIONARY = load_imagenet_mapping()

print(ORACLE_DICTIONARY)

Successfully loaded 1000 class mappings!
{'n01440764': 0, 'n01443537': 1, 'n01484850': 2, 'n01491361': 3, 'n01494475': 4, 'n01496331': 5, 'n01498041': 6, 'n01514668': 7, 'n01514859': 8, 'n01518878': 9, 'n01530575': 10, 'n01531178': 11, 'n01532829': 12, 'n01534433': 13, 'n01537544': 14, 'n01558993': 15, 'n01560419': 16, 'n01580077': 17, 'n01582220': 18, 'n01592084': 19, 'n01601694': 20, 'n01608432': 21, 'n01614925': 22, 'n01616318': 23, 'n01622779': 24, 'n01629819': 25, 'n01630670': 26, 'n01631663': 27, 'n01632458': 28, 'n01632777': 29, 'n01641577': 30, 'n01644373': 31, 'n01644900': 32, 'n01664065': 33, 'n01665541': 34, 'n01667114': 35, 'n01667778': 36, 'n01669191': 37, 'n01675722': 38, 'n01677366': 39, 'n01682714': 40, 'n01685808': 41, 'n01687978': 42, 'n01688243': 43, 'n01689811': 44, 'n01692333': 45, 'n01693334': 46, 'n01694178': 47, 'n01695060': 48, 'n01697457': 49, 'n01698640': 50, 'n01704323': 51, 'n01728572': 52, 'n01728920': 53, 'n01729322': 54, 'n01729977': 55, 'n01734418': 56,

In [ ]:
reference_dir = "D:\Recent Advances\ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\sampled_tin_no_resize2"

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\Jerry\AppData\Local\Temp\ipykernel_7568\191139112.py:1: SyntaxWarning: invalid escape sequence '\R'
  reference_dir = "D:\Recent Advances\ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\sampled_tin_no_resize2"


In [ ]:
unique_object_ids = [] #we get the complete list of images needed to create the split

# Iterate through all 200 class folders
class_folders = [f.path for f in os.scandir(reference_dir) if f.is_dir()]

for folder in class_folders:
    class_id = os.path.basename(folder) 
    
    # Grab all 5 JPEGs in this folder
    image_paths = glob.glob(os.path.join(folder, "*.JPEG"))
    
    for img_path in image_paths:
        filename = os.path.basename(img_path)
        base_name = os.path.splitext(filename)[0] # e.g., 'ILSVRC2012_val_00005450'
        
        # Create the unique identifier
        unique_id = f"{class_id}_{base_name}"
        unique_object_ids.append(unique_id)

print(f"Found {len(unique_object_ids)} unique objects.")


Found 1000 unique objects.


In [ ]:

random.seed(42) 
random.shuffle(unique_object_ids)

# 800 Train / 100 Val / 100 Test
train_ids = set(unique_object_ids[:800])
val_ids = set(unique_object_ids[800:900])
test_ids = set(unique_object_ids[900:])

print(f"Train: {len(train_ids)}, Val: {len(val_ids)} test:{len(test_ids)}")
# print(val_ids)

Train: 900, Val: 100


In [ ]:

def determine_split(filename, train_set, val_set,test_set):
   
    
    for unique_id in train_set:
        if unique_id in filename: return "train"
        
    for unique_id in val_set:
        if unique_id in filename: return "val"
    
    for unique_id in test_set:
        if unique_id in filename: return "test"
        
        
    return None



In [ ]:
ae_dir = Path(r"ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure")
param_dir = Path(r"ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\param_control")


ae_images = list(ae_dir.glob("l*/param_1/*/*.JPEG"))
print(ae_images)

[WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-diverse-test/auto_exposure/l1/param_1/n01443537/ILSVRC2012_val_00000994.JPEG'), WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-diverse-test/auto_exposure/l1/param_1/n01443537/ILSVRC2012_val_00002848.JPEG'), WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-diverse-test/auto_exposure/l1/param_1/n01443537/ILSVRC2012_val_00019459.JPEG'), WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-diverse-test/auto_exposure/l1/param_1/n01443537/ILSVRC2012_val_00038057.JPEG'), WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-diverse-test/auto_exposure/l1/param_1/n01443537/ILSVRC2012_val_00045761.JPEG'), WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-diverse-test/auto_exposure/l1/param_1/n01629819/ILSVRC2012_val_00001816.JPEG'), WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-diverse-test/auto_exposure/l1/param_1/n01629819/ILSVRC2012_val_00029763.JPEG'), WindowsPath('ImageNet-ES-Diverse/ImageNet-ES-Di

In [ ]:


ae_path = Path(r"ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00000994.JPEG")

wordnet_id = ae_path.parts[-2]
print(f"Extracted WordNet ID: {wordnet_id}")

# This will now successfully find 'n01443537' in the dictionary
true_class_index = ORACLE_DICTIONARY[wordnet_id]
print(f"ResNet True Class Index: {true_class_index}")


Extracted WordNet ID: n01443537
ResNet True Class Index: 1


In [ ]:
def find_best_parameter(ae_path, param_dir, model, transform, device):
    """
    Finds the manual camera parameter (1-27) that yields the highest ResNet confidence.
    Uses tensor batching to maximize GPU utilization.
    """
    # 1. Extract metadata from the AE path to locate the manual shots
    light_env = ae_path.parts[-4]    # e.g., 'l1'
    class_id = ae_path.parts[-2]     # e.g., 'n01443537'
    filename = ae_path.name          # e.g., 'ILSVRC2012_val_00000994.JPEG'
    
    valid_indices = []
    image_tensors = []
    
    for p_idx in range(1, 28):
        manual_folder = f"param_{p_idx}"
        manual_img_path = param_dir / light_env / manual_folder / class_id / filename
        
        if manual_img_path.exists():
      
            img = Image.open(manual_img_path).convert('RGB')
            tensor = transform(img)
            image_tensors.append(tensor)
            valid_indices.append(p_idx)
            
   
    if not image_tensors:
        return -1, -1.0 

    batch_tensor = torch.stack(image_tensors).to(device)
    true_class_index = ORACLE_DICTIONARY[class_id]
    # 4. Perform a single forward pass on the GPU
    with torch.no_grad():
        logits = model(batch_tensor)
        probabilities = F.softmax(logits, dim=1) # (N, 1000) (27 x 1000)

        # print(probabilities.shape)
        
        oracle_confidences = probabilities[:, true_class_index] # og class - 150 - 2%

        # print(f"Oracle confidence --> {oracle_confidences} --> shape {oracle_confidences.shape}")
        # print(oracle_confidences[19])
        # print(torch.argmax(oracle_confidences))
        best_batch_idx = torch.argmax(oracle_confidences).item() 
        # print(f"best batch: {best_batch_idx}" )
        
        # --- WHAT DID RESNET ACTUALLY GUESS? ---
        # dim=1 tells PyTorch to look across all 1000 classes and find the highest one
        #to check whether the resnet has predicted the correct class instead of another class (confirmation)
        predicted_classes = torch.argmax(probabilities, dim=1) #97% accuracy  - 327
      
        
        # Extract the specific guess ResNet made for our winning camera parameter
        resnet_winning_guess = predicted_classes[best_batch_idx].item() #327 - best parameter

        
        
    # 5. Map the batch index back to the actual parameter index (1-27)
    best_p_idx = valid_indices[best_batch_idx]
    highest_confidence = oracle_confidences[best_batch_idx].item()
    # print(f"confidence --> {highest_confidence} index --> {best_p_idx}")
    
    # Return all THREE pieces of information now!
    return best_p_idx, highest_confidence, resnet_winning_guess

In [ ]:
train_data = []
val_data = []
test_data = []

ae_dir = Path(r"ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure")
param_dir = Path(r"ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\param_control")


ae_images = list(ae_dir.glob("l*/param_1/*/*.JPEG")) #only taking params1 as it is the same image in all params

print(f"Found {len(ae_images)} Auto-Exposure scenes to process...")




#for all the images in the list 
for ae_path in ae_images:
    
    #extract all the details to get the same image with different params from the parameter dataset
    light_env = ae_path.parts[-4]    
    class_id = ae_path.parts[-2]     
    filename = ae_path.name         
    base_name = ae_path.stem         
    
    
    unique_id = f"{class_id}_{base_name}"
    true_class_index = ORACLE_DICTIONARY[class_id]
    # print(unique_id)
    
    #check where that image belongs ie, in train or val or test
    target_split = determine_split(unique_id, train_ids, val_ids,test_ids)
    if target_split is None:
        continue

        
        
      
    best_param_index, confidence, resnet_guess = find_best_parameter(
    ae_path=ae_path, 
    param_dir=param_dir, 
    model=model, 
    transform=preprocess, 
    device=device
)  
    
    
    
    
    
        
# --- C. Save the result ---
    if best_param_index != -1:
        row = {
            "ae_image_path": str(ae_path),
            "light_environment": light_env,
            "best_param_index": best_param_index - 1, 
            "confidence_score": confidence,
            "true_class_index": true_class_index,
            "resnet_predicted_index": resnet_guess # Save ResNet's guess to CSV too!
        }

        # print(row)
        
        if target_split == "train": train_data.append(row)
        elif target_split == "val": val_data.append(row)

        pd.DataFrame(train_data).to_csv("lens_train.csv", index=False)
        pd.DataFrame(val_data).to_csv("lens_val.csv", index=False)

        print(f"{len(train_data)+len(val_data)} => ae_path {ae_path} confidence {confidence} pindex {best_param_index} in split {target_split} resnet guess index {resnet_guess} --> correct index{true_class_index}")

print(f"training data:{train_data[0]} \n ")



print("Batch saving complete.")

Found 6000 Auto-Exposure scenes to process...
1 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00000994.JPEG confidence 0.978685736656189 pindex 20 in split train resnet guess index 1 --> correct index1
2 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00002848.JPEG confidence 0.9451826214790344 pindex 20 in split train resnet guess index 1 --> correct index1
3 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00019459.JPEG confidence 0.3679666817188263 pindex 14 in split train resnet guess index 1 --> correct index1
4 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00038057.JPEG confidence 0.046141430735588074 pindex 19 in split train resnet guess index 1 --> correct index1
5 => ae_path ImageNet-ES-Diverse\ImageNet

In [ ]:
print(row)

{'ae_image_path': 'ImageNet-ES-Diverse\\ImageNet-ES-Diverse\\es-diverse-test\\auto_exposure\\l7\\param_1\\n12267677\\ILSVRC2012_val_00048992.JPEG', 'light_environment': 'l7', 'best_param_index': 0, 'confidence_score': 0.25968286395072937, 'true_class_index': 988, 'resnet_predicted_index': 988}


1 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00000994.JPEG confidence 0.978685736656189 pindex 20 in split train resnet guess index 1 --> correct index1
2 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00002848.JPEG confidence 0.9451826214790344 pindex 20 in split train resnet guess index 1 --> correct index1
3 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00019459.JPEG confidence 0.3679666817188263 pindex 14 in split train resnet guess index 1 --> correct index1
4 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01443537\ILSVRC2012_val_00045761.JPEG confidence 0.049480367451906204 pindex 5 in split train resnet guess index 327 --> correct index1
5 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01629819\ILSVRC2012_val_00001816.JPEG confidence 0.29981711506843567 pindex 15 in split train resnet guess index 28 --> correct index25
6 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01629819\ILSVRC2012_val_00029763.JPEG confidence 0.4312315583229065 pindex 1 in split val resnet guess index 25 --> correct index25
7 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01629819\ILSVRC2012_val_00039753.JPEG confidence 0.433806449174881 pindex 11 in split train resnet guess index 25 --> correct index25
8 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01629819\ILSVRC2012_val_00046243.JPEG confidence 0.7109512090682983 pindex 11 in split train resnet guess index 25 --> correct index25
9 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01629819\ILSVRC2012_val_00046985.JPEG confidence 0.02782202884554863 pindex 10 in split train resnet guess index 26 --> correct index25
10 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01641577\ILSVRC2012_val_00016782.JPEG confidence 0.02048947662115097 pindex 10 in split train resnet guess index 29 --> correct index30
11 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01641577\ILSVRC2012_val_00027869.JPEG confidence 0.15686053037643433 pindex 1 in split train resnet guess index 32 --> correct index30
12 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01641577\ILSVRC2012_val_00043312.JPEG confidence 0.09464416652917862 pindex 11 in split train resnet guess index 32 --> correct index30
13 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01641577\ILSVRC2012_val_00049119.JPEG confidence 0.05862916633486748 pindex 1 in split train resnet guess index 36 --> correct index30
14 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01644900\ILSVRC2012_val_00005450.JPEG confidence 0.2861092686653137 pindex 11 in split train resnet guess index 32 --> correct index32
15 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01644900\ILSVRC2012_val_00032415.JPEG confidence 0.050440121442079544 pindex 11 in split train resnet guess index 317 --> correct index32
16 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01644900\ILSVRC2012_val_00033769.JPEG confidence 0.003724584821611643 pindex 10 in split train resnet guess index 344 --> correct index32
17 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01644900\ILSVRC2012_val_00036942.JPEG confidence 0.6540940999984741 pindex 1 in split train resnet guess index 32 --> correct index32
18 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01644900\ILSVRC2012_val_00041919.JPEG confidence 0.00506629841402173 pindex 11 in split train resnet guess index 29 --> correct index32
19 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01698640\ILSVRC2012_val_00004402.JPEG confidence 0.024720270186662674 pindex 14 in split train resnet guess index 29 --> correct index50
20 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01698640\ILSVRC2012_val_00009083.JPEG confidence 0.007094245869666338 pindex 10 in split train resnet guess index 395 --> correct index50
21 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01698640\ILSVRC2012_val_00037268.JPEG confidence 0.007039207965135574 pindex 1 in split train resnet guess index 346 --> correct index50
22 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01698640\ILSVRC2012_val_00038620.JPEG confidence 0.005596603732556105 pindex 14 in split train resnet guess index 344 --> correct index50
23 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01698640\ILSVRC2012_val_00047405.JPEG confidence 0.03162866085767746 pindex 1 in split train resnet guess index 735 --> correct index50
24 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01742172\ILSVRC2012_val_00013730.JPEG confidence 0.17047031223773956 pindex 10 in split val resnet guess index 61 --> correct index61
25 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01742172\ILSVRC2012_val_00018456.JPEG confidence 0.5919163823127747 pindex 10 in split train resnet guess index 61 --> correct index61
26 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01742172\ILSVRC2012_val_00018622.JPEG confidence 0.252347856760025 pindex 10 in split val resnet guess index 61 --> correct index61
27 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01742172\ILSVRC2012_val_00018907.JPEG confidence 0.1903199702501297 pindex 1 in split train resnet guess index 61 --> correct index61
28 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01742172\ILSVRC2012_val_00031749.JPEG confidence 0.08287806063890457 pindex 20 in split train resnet guess index 61 --> correct index61
29 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01768244\ILSVRC2012_val_00012521.JPEG confidence 0.5380007028579712 pindex 20 in split val resnet guess index 69 --> correct index69
30 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01768244\ILSVRC2012_val_00021025.JPEG confidence 0.05960066244006157 pindex 1 in split train resnet guess index 5 --> correct index69
31 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01768244\ILSVRC2012_val_00024601.JPEG confidence 0.12511380016803741 pindex 1 in split val resnet guess index 911 --> correct index69
32 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01768244\ILSVRC2012_val_00035606.JPEG confidence 0.0610889308154583 pindex 1 in split train resnet guess index 69 --> correct index69
33 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01768244\ILSVRC2012_val_00040047.JPEG confidence 0.3984183669090271 pindex 20 in split train resnet guess index 69 --> correct index69
34 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01770393\ILSVRC2012_val_00001887.JPEG confidence 0.4630650281906128 pindex 15 in split train resnet guess index 71 --> correct index71
35 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01770393\ILSVRC2012_val_00005821.JPEG confidence 0.13682638108730316 pindex 15 in split train resnet guess index 71 --> correct index71
36 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01770393\ILSVRC2012_val_00033658.JPEG confidence 0.12984031438827515 pindex 10 in split train resnet guess index 76 --> correct index71
37 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01770393\ILSVRC2012_val_00043619.JPEG confidence 0.9836443066596985 pindex 15 in split train resnet guess index 71 --> correct index71
38 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01770393\ILSVRC2012_val_00046299.JPEG confidence 0.878546416759491 pindex 15 in split train resnet guess index 71 --> correct index71
39 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774384\ILSVRC2012_val_00006437.JPEG confidence 0.9846858978271484 pindex 10 in split train resnet guess index 75 --> correct index75
40 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774384\ILSVRC2012_val_00008718.JPEG confidence 0.8151835203170776 pindex 5 in split train resnet guess index 75 --> correct index75
41 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774384\ILSVRC2012_val_00025455.JPEG confidence 0.015494964085519314 pindex 21 in split train resnet guess index 657 --> correct index75
42 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774384\ILSVRC2012_val_00025508.JPEG confidence 0.9820641279220581 pindex 5 in split train resnet guess index 75 --> correct index75
43 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774384\ILSVRC2012_val_00041127.JPEG confidence 0.09049920737743378 pindex 14 in split val resnet guess index 126 --> correct index75
44 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774750\ILSVRC2012_val_00032091.JPEG confidence 0.8937690258026123 pindex 1 in split train resnet guess index 76 --> correct index76
45 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774750\ILSVRC2012_val_00044170.JPEG confidence 0.27038538455963135 pindex 10 in split val resnet guess index 76 --> correct index76
46 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774750\ILSVRC2012_val_00045733.JPEG confidence 0.6437510848045349 pindex 1 in split train resnet guess index 76 --> correct index76
47 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01774750\ILSVRC2012_val_00048505.JPEG confidence 0.13306564092636108 pindex 20 in split train resnet guess index 120 --> correct index76
48 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01784675\ILSVRC2012_val_00014858.JPEG confidence 0.014378736726939678 pindex 1 in split train resnet guess index 29 --> correct index79
49 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01784675\ILSVRC2012_val_00020456.JPEG confidence 0.06479140371084213 pindex 1 in split train resnet guess index 111 --> correct index79
50 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01784675\ILSVRC2012_val_00022081.JPEG confidence 0.026295460760593414 pindex 10 in split train resnet guess index 111 --> correct index79
51 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01784675\ILSVRC2012_val_00038504.JPEG confidence 0.03788303956389427 pindex 5 in split train resnet guess index 116 --> correct index79
52 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01784675\ILSVRC2012_val_00038833.JPEG confidence 0.9901636242866516 pindex 6 in split train resnet guess index 79 --> correct index79
53 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01855672\ILSVRC2012_val_00004235.JPEG confidence 0.021023785695433617 pindex 10 in split train resnet guess index 87 --> correct index99
54 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01855672\ILSVRC2012_val_00009610.JPEG confidence 0.03816887363791466 pindex 11 in split train resnet guess index 974 --> correct index99
55 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01855672\ILSVRC2012_val_00025256.JPEG confidence 0.01319611445069313 pindex 10 in split train resnet guess index 342 --> correct index99
56 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01855672\ILSVRC2012_val_00032295.JPEG confidence 0.17648155987262726 pindex 20 in split train resnet guess index 98 --> correct index99
57 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01882714\ILSVRC2012_val_00023031.JPEG confidence 0.01842142641544342 pindex 2 in split train resnet guess index 181 --> correct index105
58 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01882714\ILSVRC2012_val_00026297.JPEG confidence 0.8731310367584229 pindex 1 in split train resnet guess index 105 --> correct index105
59 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01882714\ILSVRC2012_val_00027599.JPEG confidence 0.09837091714143753 pindex 1 in split train resnet guess index 181 --> correct index105
60 => ae_path ImageNet-ES-Diverse\ImageNet-ES-Diverse\es-diverse-test\auto_exposure\l1\param_1\n01882714\ILSVRC2012_val_00034332.JPEG confidence 0.7051548361778259 pindex 1 in split train resnet guess index 105 --> correct index105